**Instructions:**

- Do not write your name on the assignment. Be careful about any warnings that might display some file path with your name included.
- Receiving help from classmates and/or ideas from Generative AI is allowed. **However, you must submit your own original work.**
- For questions that require coding, you need to write the relevant code and display its output. Your output should either be the direct answer to the question or clearly display the answer in it.
- For questions that require a written answer (sometimes along with the code), you need to put your answer in a Markdown cell. Writing the answer as a comment or as a print line is not acceptable.
- You need to render this file as HTML using Quarto and submit the HTML file. **Please note that this is a requirement and not optional.** A submission cannot be graded until it is properly rendered.

Import all the libraries and tools you need below.

In [61]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score
import numpy as np

## 1)

In this exercise set, you will use the **creditcard.csv** file. Each observation is a credit card transaction. Most of the variables are created by a dimensionality reduction method called Principle Component Analysis (PCA), which will be covered in Week 5 or 6. The `Class` values are stored in the last variable and represent whether the transaction is a regular (0) or fraudulent (1) transaction.

### a)

Read the data. Print the number of Class 0 and Class 1 observations. You should see that there is a class imbalance. **(5 points)**

In [62]:
df = pd.read_csv('creditcard.csv')

zero = len(df[df["Class"] == 0])
one = len(df[df["Class"] == 1])

print(f"zero class samples: {zero}")
print(f"one class samples: {one}")

zero class samples: 284315
one class samples: 492


### b)

There are different methods to handle the class imbalance in the dataset at hand. In this set, you will undersample the majority class.

There are built-in functions for undersampling in specialized libraries, such as [imblearn](https://imbalanced-learn.org/stable/), which is not installed in any Python environment by default. To make this step reproducible, you will use a more low-level approach with pandas:

- Separate the regular (0) and fraudulent (1) observations into two different DataFrames by filtering.
- Sample 1000 observations from the DataFrame with the majority class. Use `.sample` method with `random_state=2`.
- Concatenate the undersampled majority class DataFrame and the minority class DataFrame.

**(10 points)**

In [63]:
df_zero = df[df["Class"] == 0]
df_one = df[df["Class"] == 1]

df_zero_sample = df_zero.sample(n=1000, random_state=2)
df_balanced = pd.concat([df_one, df_zero_sample])

# print(df_balanced["Class"].value_counts())

### c)

Drop the `Amount` and `Time` variables. The rest of the variables (except `Class`) are the features.

Create the training and test sets with a 70%-30% split and `random_state=0`. **Stratify the data.**

**Note:** Since the PCA-created feature values are all in the same order of magnitude, scaling is not necessary.

**(5 points)**

In [64]:
df_balanced = df_balanced.drop(columns=["Amount", "Time"])

X = df_balanced.drop(columns = 'Class')
y = df_balanced["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state = 0,
    stratify = y
)

## 2)

After preprocessing the data, you will tune and train a **linear** Support Vector Machine (SVM) classifier.

### a)

Create a **linear** SVM classifier. Its training algorithm has some random processes, so keep `random_state=1`. You will need some prediction probabilities, which is not in the original training algorithm of an SVM. To use the extended version of the algorithm, use `probability=True`. **(5 points)**

**Note:** Use `SVC`, you will use `LinearSVC` in the next in-class assignment.

In [65]:
svm = SVC(kernel='linear', random_state=1, probability=True)

### b)

Tune the model with the following specifications:

- For the grid of the **only hyperparameter**, use $10^i$ where i is an integer between -4 and 3 (inclusive).
- Keep your folds **stratified**. In the same object, also `shuffle` the data with `random_state=15`.
- Use both accuracy and recall as your scoring metric, but recall should be the primary (`refit`) metric to find the best hyperparameter.

Print the best hyperparameter value and the cross-validation (CV) recall score. (Both can be returned with the attributes of a `GridSearchCV` object.)

**Note:** This should take 3-4 minutes to run. You can use the given lines to keep track of the elapsed time. (You do not have to and you can delete those lines if you wish.)

**(20 points)**

In [66]:
tic = time.time()

########## YOUR CODE HERE #############

C_options = [10**i for i in range(-4, 4)]
param_grid = {"C": C_options}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=15
)

grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=cv,
    scoring=["accuracy", "recall"],
    refit="recall",
    n_jobs=-1 # using more computing power to make it faster :)
)

grid_search.fit(X_train, y_train)

print("Best C:", grid_search.best_params_["C"])
print("Best CV Recall:", grid_search.best_score_)

#######################################

toc = time.time()
print('Elapsed Time: ', (toc-tic)/60, 'minutes.')

Best C: 1000
Best CV Recall: 0.9013213981244672
Elapsed Time:  1.3321985840797423 minutes.


### c)

Print the `cv_results_` as a DataFrame to get more insight about the cross-validation process. Print only three columns: (1) the hyperparameter values, (2) the average cross-validation accuracy and (3) the average cross-validation recall.

- Did the model sacrifice some accuracy to maximize the recall or is the accuracy increasing with the recall?
- What probability threshold do these accuracy and recall values correspond to?

**Note:** The column names can be a bit misleading - there are not any test results here, only average cross-validation results.

**(10 points)**

In [67]:
results = pd.DataFrame(grid_search.cv_results_)

results[["param_C", "mean_test_accuracy", "mean_test_recall"]]

,param_C,mean_test_accuracy,mean_test_recall
0,0.0001,0.926265,0.776343
1,0.001,0.939676,0.817050
2,0.01,0.948293,0.857715
3,0.1,0.947336,0.866411
4,1,0.951173,0.880946
5,10,0.954053,0.895524
6,100,0.955010,0.895524
7,1000,0.956924,0.901321


1) Accuracy increases alongside recall as C increases. Accuracy went from 92.6% to 95.7% and recall went from just 77.6% to 90.1%. In this case, maximizing recall did not sacrificed accuracy. The only case in which accuracy decreased was when C went from 0.01 to 0.1.

2) The scores are based on SVM's default rule, which uses a threshold of 0. They do not necessarily correspond to a probability threshold of 0.5 even if we enabled probability estimates.

## 3)

For a classification model, tuning the model hyperparameters is not enough. The decision threshold needs to be tuned as well, especially for tasks like fraud detection.

### a)

- For the classification task in this set, are False Negatives and False Positives equally important?
- If not, which one is more important to avoid? Why?
- Which classification metric needs to be maximized (ideally)?

**(5 points)**

1) No, they are both important but which one is more important depends of the context and what the bank is trying to achieve. If the bank is prioritizing customer satisfaction and customer service, then False Positives are more important to avoid. If the bank is prioritizing financial loss, then False Negatives are more important to avoid.

2) In my opinion, False Negatives are way more important to avoid because the bank would lose money if they let a fradulent transaction go. If a customer is wrongly accused of fraud, it is a big inconvenience for the customer but it is not as bad as losing money.

3) Recall, because it measures the proportion of actual fraudulent transactions that the model correctly identifies.

### b)

In order to tune the decision threshold, you need to find the cross-validation accuracy and recall for all possible threshold values (with a reasonable granularity).

- Start by creating an empty DataFrame with three columns: `thr`, `acc`, and `rec`. You will use this DataFrame to store your results.
- Initialize a `counter` variable at 0. You will use this to index the DataFrame to store your results.
- Using the `best_estimator_` of the cross-validation in Question 2b, obtain the cross-validation Class 1 probabilities of all observations. You need the `cross_val_predict` function for this; please consider checking the posted notes if you are not familiar with its usage. You also need to keep the same `cv` input as in Question 2, so the results are consistent.
- For each threshold value between 0 and 1, with a stepsize of 0.01, calculate the accuracy and recall score when the Class 1 probabilities are converted to class values with the threshold value.
- Store the accuracy and recall values of all thresholds in your DataFrame.

**Note:** This should take 3-4 minutes to run. You can use the given lines to keep track of the elapsed time. (You do not have to and you can delete those lines if you wish.)

**(20 points)**

In [68]:
tic = time.time()

########## YOUR CODE HERE #############

empty_df = pd.DataFrame(columns=["thr", "acc", "rec"])
counter = 0

best_svm = grid_search.best_estimator_

cv_probs = cross_val_predict(
    best_svm,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)

fraud_probs = cv_probs[:, 1]

for threshold in np.arange(0, 1.01, 0.01):

    predictions = (fraud_probs >= threshold).astype(int)

    accuracy = accuracy_score(y_train, predictions)
    recall = recall_score(y_train, predictions)

    empty_df.loc[counter] = [threshold, accuracy, recall]
    counter += 1
    
print(empty_df)


#######################################

toc = time.time()
print('Elapsed Time: ', (toc-tic)/60, 'minutes.')

      thr       acc       rec
0    0.00  0.329502  1.000000
1    0.01  0.329502  1.000000
2    0.02  0.329502  1.000000
3    0.03  0.329502  1.000000
4    0.04  0.329502  1.000000
..    ...       ...       ...
96   0.96  0.924330  0.770349
97   0.97  0.921456  0.761628
98   0.98  0.921456  0.761628
99   0.99  0.903257  0.706395
100  1.00  0.670498  0.000000

[101 rows x 3 columns]
Elapsed Time:  0.37120463450749713 minutes.


### c)

Print all the rows with threshold values that return a perfect recall. Is there a threshold value that returns a reasonable accuracy with the perfect recall? (The word "reasonable" sounds subjective, but you should see a clear difference from the accuracy results in Question 2.)

**(5 points)**

In [69]:
perfect_recall = empty_df[empty_df["rec"] == 1.0]

# best_perfect_recall = perfect_recall.loc[
#     perfect_recall["acc"].idxmax()
# ]

# best_perfect_recall

perfect_recall

,thr,acc,rec
0,0.00,0.329502,1.0
1,0.01,0.329502,1.0
2,0.02,0.329502,1.0
3,0.03,0.329502,1.0
4,0.04,0.329502,1.0
5,0.05,0.329502,1.0
6,0.06,0.329502,1.0
7,0.07,0.331418,1.0
8,0.08,0.334291,1.0
9,0.09,0.397510,1.0


No, there is no threshold that achieves perfect recall with reasonable accuracy. The highest accuracy with 100% recall is approximately 39.75%.

### d)

Find the threshold value that returns the highest recall while the accuracy is above 80%. This will be your tuned threshold. **(5 points)**

In [70]:
accuracy_over_eighty = empty_df[empty_df["acc"] > .8]

accuracy_over_eighty.sort_values(by="rec", ascending=False)

best_row = accuracy_over_eighty.loc[
    accuracy_over_eighty["rec"].idxmax()
]

tuned_threshold = best_row["thr"]

print(f"Tuned threshold: {tuned_threshold}")


Tuned threshold: 0.1


### e)

Using the tuned model from Question 2 and the tuned threshold from Part d in this question, find the test accuracy and recall. Do you observe any overfitting? (This conclusion might be up for debate, so just write what you observe and how you interpret it.)

**(10 points)**

In [71]:
best_svm = grid_search.best_estimator_

test_probs = best_svm.predict_proba(X_test)[:, 1]

y_pred_test = (test_probs >= tuned_threshold).astype(int)

test_accuracy = accuracy_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)

print("Test Accuracy:", test_accuracy)
print("Test Recall:", test_recall)

Test Accuracy: 0.9486607142857143
Test Recall: 0.8445945945945946


The test accuracy was 94.9% and the test recall was 84.5%. Compared to the cross-validation results, accuracy increased from 80.94% to 94.87%, while recall only decreased from 86.9% to 84.5%.

I don't think there is strong evidence of overfitting because the test accuracy is actually way higher and the recall only decreased by around 2.5 percentage points. However, the large difference in accuracy is interesting and suggests the model performed differently on the test data. Overall, the model generalizes well, although it misses a more fraudulent transactions than during cross validation.